This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path
from enum import Enum

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide, cut_low_l, UnfoldingProcessInfo

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
    1.2,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.01 MeVee)",
    0.01,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

In [ ]:
# matrix_dir_name = "response_matrix_hi_res"
# matrix_dir_name = "response_matrix_4k"
# matrix_dir_name = "response_matrix_50keV_sigma"
# matrix_dir_name = "response_matrix_50keV_sigma_NoFWHM"
# matrix_dir_name = "response_matrix_R3"
# matrix_dir_name = "response_matrix_R4_sigma_50keV"
# matrix_dir_name = "response_matrix_R4_sigma_50keV_FWHM"
matrix_dir_name = "response_matrix_R4_mono"

# phds_to_compare = [
#     "E1.csv.npy",
#     "E2.csv.npy",
#     "E3.csv.npy",
# ]
# labels = ["E1", "E2", "E3"]
phds_to_compare = ["sigma_50keV_FWHM.csv.npy"]
labels = ["DD fusion"]
# phds_to_compare = [
#     "neutron_0.496_MeV.csv.npy",
#     "neutron_1.054_MeV.csv.npy",
#     "neutron_1.550_MeV.csv.npy",
#     "neutron_2.046_MeV.csv.npy",
#     "neutron_2.480_MeV.csv.npy",
#     "neutron_3.038_MeV.csv.npy",
#     "neutron_3.534_MeV.csv.npy",
# ]
# labels = [
#     "0.496 MeV",
#     "1.054 MeV",
#     "1.550 MeV",
#     "2.046 MeV",
#     "2.480 MeV",
#     "3.038 MeV",
#     "3.534 MeV",
# ]

In [ ]:
R = load_neutron_response_matrix(
    Path(matrix_dir_name),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
# R_sum = R.counts.sum(axis=0, keepdims=True)
# R_normed = R.counts / R_sum
# R = NDHistogram(R_normed, R.midpoints)

In [ ]:
R.counts.shape

In [ ]:
np.sum(R.counts, axis=1)

In [ ]:
R.counts[:, 20]

In [ ]:
base_path = Path("unfolding_test")
# base_path = Path("response_matrix_R4_mono")
bins = np.arange(bins_min, bins_max + bins_width, bins_width)
Ns = []
# test_file = base_path / "output_2.45 MeV.txt"
# test_file = base_path / "sim_2.450_MeV_Tbird_NoRes_Bin1000.csv.npy"
# test_file = base_path / "mono.csv.npy"
# test_file = base_path / "mono_FWHM.csv.npy"
# test_file = base_path / "E1.csv.npy"
for filename in phds_to_compare:
    data_file = base_path / filename
    L_array = np.load(data_file)
    
    np_cps, *_ = np.histogram(L_array, bins=bins)
    np_cps = np_cps.reshape(-1, 1)
    np_Ls = (bins[1:] + bins[:-1]) / 2

    # np_cps_sum = np_cps.sum()
    # np_cps_normed = np_cps / np_cps_sum
    np_cps_normed = np_cps

    N = NDHistogram(np_cps_normed, [np_Ls, np.ones(1)])
    Ns.append(N)

In [ ]:
Nphis = []
for N in Ns:
    phi, _ = unfold_spectrum(
        R,
        N,
        L_cut=0.05,
        # tolerance=0.0000001,
        max_iterations=1000,
    )
    Nphis.append(phi)

In [ ]:
Nphis_flat = [phi.counts.reshape(-1) for phi in Nphis]
Nphis_mids = [phi.midpoints[1] for phi in Nphis]

In [ ]:
figsize=(9,6)
dd_scaling = 1
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
for phi_mids, phi_flat, label in zip(Nphis_mids, Nphis_flat, labels):
    ax.plot(
        phi_mids, phi_flat, marker="o", markersize=3,
        # label="2.45 MeV monoenergetic simulation"
        # label="E1"
        label=label
    )
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("E (MeV)", fontsize=fontsize)
ax.set_ylabel("Normalized counts", fontsize=fontsize)
ax.legend()
# ax.set_yscale("log")
ax.tick_params(labelsize=fontsize)
plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

In [ ]:
# weights = unfold_info["weights"]
# W_L_mids, W_phi_mids = weights[0].midpoints
# print(W_L_mids.shape)
# print(W_phi_mids.shape)
# print(weights[0].shape)

L_mids, phi_mids = R.midpoints
L_widths = (L_mids[1:] - L_mids[:-1])
L_widths = np.insert(L_widths, (0,), L_widths[0])
phi_widths = phi_mids[1:] - phi_mids[:-1]
phi_widths = np.insert(phi_widths, (0,), phi_widths[0])

_x = phi_mids - (phi_widths / 2)
_y = L_mids - (L_widths / 2)
_xx, _yy = np.meshgrid(_x, _y)
x, y = np.ravel(_xx), np.ravel(_yy)

_xxw, _yyw = np.meshgrid(phi_widths, L_widths)
xw, yw = np.ravel(_xxw), np.ravel(_yyw)

# tops = [np.ravel(weight.counts) for weight in weights]
top = np.ravel(R.counts)
with np.errstate(invalid="ignore", divide="ignore"):
    top = np.nan_to_num(np.log10(top), nan=np.nan, posinf=np.nan, neginf=np.nan)
bottom = np.zeros_like(top)

# Zmax_list = [np.nanmax(top) for top in tops]
Zmax = max(top)
Zmin = 0

In [ ]:
import matplotlib as mpl

vaporwave_colors = [
    [255, 255, 255],
    [128, 69, 229],
    [75,127,255],
    [0,255,255],
    [255,186,129],
    [255,209,86],
    [252,120,183]
]
vaporwave_colors = [[value/255 for value in color] for color in vaporwave_colors]
vaporwave = mpl.colors.LinearSegmentedColormap.from_list("vaporwave", vaporwave_colors, N=256)
# vaporwave = vaporwave.resampled(256)
vaporwave

In [ ]:
norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)
sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
sc.set_array([])

# weights_colors = [vaporwave(norm(top)) for top in tops]
R_color = vaporwave(norm(top))

In [ ]:
# fig, ax = plt.subplots(figsize=figsize, projection="3d")
figsize=(9,6)
fig = plt.figure(figsize=figsize, dpi=300)
ax = fig.add_subplot(projection="3d")
ax.view_init(elev=30, azim=45)

# artists = []
# _frame = 0
# for i in range(showevery_weight-1, len(tops), showevery_weight):
#     print(f"{_frame % 10}", end="")
#     _frame += 1
#     top = tops[i]
#     color = weights_colors[i]
nan_mask = ~np.isnan(top)
# masked_args = [param[nan_mask] for param in [x, y, bottom, xw, yw, top]]
masked_args = [param[nan_mask] for param in [x, y, bottom, xw, yw, top]]
masked_color = R_color[nan_mask]
# W_plot = ax.bar3d(x, y, bottom, xw, yw, top, color=color)
ax.bar3d(*masked_args, color=masked_color)

# ax.set_zlim(0, 0.25)
# frame = ax.annotate(f"Frame {i+1}", (1, 1), xycoords="axes fraction", horizontalalignment="right", verticalalignment="top")
# artists.append([W_plot, frame])
plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

In [ ]:
# TODO finish brute force method
def safe_divide(a, b):
    try:
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.nan_to_num(a / b, nan=np.nan, posinf=np.nan, neginf=np.nan)
    except ZeroDivisionError:
        return np.nan


def r_phi_sum(r, phi):
    _m, _n = r.shape
    result = np.zeros((_m, 1))

    for i in range(_m):
        j_sum = 0
        for j in range(_n):
            product = r[i][j] * phi[0][j]
            if not np.isnan(product):
                j_sum += product
        result[i][0] = j_sum

    return result


def weight_looped(r, n, phi, sigma):
    _m, _n = r.shape
    result = np.zeros((_m, _n))
    _r_phi_sum = r_phi_sum(r, phi)

    for i in range(_m):
        for j in range(_n):
            l_numer = r[i][j] * phi[0][j]
            l_denom = _r_phi_sum[i][0]
            r_numer = np.square(n[i][0])
            r_denom = np.square(sigma[i][0])
            l_frac = safe_divide(l_numer, l_denom)
            r_frac = safe_divide(r_numer, r_denom)
            result[i][j] = l_frac * r_frac
    return result


def next_phi_looped(r, n, phi, sigma):
    _m, _n = r.shape
    result = np.zeros((1, _n))
    w = weight_looped(r, n, phi, sigma)
    _r_phi_sum = r_phi_sum(r, phi)

    for j in range(_n):
        numer_i_sum = 0
        denom_i_sum = 0
        for i in range(_m):
            _w = w[i][j]
            ln_frac = safe_divide(n[i][0], _r_phi_sum[i][0])
            with np.errstate(divide="ignore", invalid="ignore"):
                ln_result = np.nan_to_num(np.log(ln_frac), nan=np.nan, posinf=np.nan, neginf=np.nan)
            numer_product = _w * ln_result
            if not np.isnan(numer_product):
                numer_i_sum += numer_product
            if not np.isnan(_w):
                denom_i_sum += _w
        exp_frac = safe_divide(numer_i_sum, denom_i_sum)
        with np.errstate(divide="ignore", invalid="ignore"):
            exp_result = np.nan_to_num(np.exp(exp_frac), nan=np.nan, posinf=np.nan, neginf=np.nan)
        result[0][j] = phi[0][j] * exp_result

    return result


def stopping_criteria_looped(r, n, phi, sigma):
    _m, _n = r.shape
    DOF = (_m-1)*(_n-1)
    _r_phi_sum = r_phi_sum(r, phi)

    i_sum = 0
    for i in range(_m):
        delta = _r_phi_sum[i][0] - n[i][0]
        d_sq = np.square(delta)
        frac = safe_divide(d_sq, np.square(sigma[i][0]))
        if not np.isnan(frac):
            i_sum += frac
    return i_sum / DOF


def unfold_spectrum_looped(r, n, phi0=None, sigma=None, L_cut=None, tolerance=0.01, max_iters=500):
    if phi0 is None:
        phi0 = NDHistogram(np.ones((1, r.shape[1])), [np.ones(1), r.midpoints[1]])
    if sigma is None:
        sigma = NDHistogram(np.sqrt(n.counts), n.midpoints)

    _r, _n, _sigma = cut_low_l(r, n, sigma=sigma, L_cut=L_cut)

    r_counts = _r.counts.copy()
    n_counts = _n.counts.copy()
    sigma_counts = _sigma.counts.copy()
    phi_counts = phi0.counts.copy()

    iters = 0
    chis = []
    phis = []
    weights = []
    errors = []
    iter_text_len = len(str(max_iters))
    
    chi_n = stopping_criteria_looped(r_counts, n_counts, phi_counts, sigma_counts)
    chi_last = chi_n
    delta_chi_last = 1
    delta_delta = 1
    
    while delta_delta > tolerance:
        _w = weight_looped(r_counts, n_counts, phi_counts, sigma_counts)
        phi_counts = next_phi_looped(r_counts, n_counts, phi_counts, sigma_counts)
        chi_n = stopping_criteria_looped(r_counts, n_counts, phi_counts, sigma_counts)
    
        delta_chi = chi_n - chi_last
        delta_delta = abs(delta_chi - delta_chi_last)
        chi_last = chi_n
        delta_chi_last = delta_chi
    
        chis.append(chi_n)
        phis.append(phi_counts)
        weights.append(_w)
        errors.append(delta_delta)
    
        if iters % 10 == 0:
            print(
                f"Iter. {iters: {iter_text_len}d}: chi = {chi_n:.3g}, rel_rate = {delta_delta: .3g}"
            )
        iters += 1
        if iters >= max_iters:
            break

    unfolding_info = UnfoldingProcessInfo(errors=errors, chis=chis, phis=phis, weights=weights)
    return phi_counts, unfolding_info

In [ ]:
testr = np.array([list(range(i, i+5)) for i in range(1,7)])
testphi = np.array([[1, 2, 3, 4, 5]])
_r_phi_sum = r_phi_sum(testr, testphi)
print(_r_phi_sum)

In [ ]:
testn = np.array([[i] for i in range(3, 21, 3)])
testsigma = np.sqrt(testn)

_m, _n = testr.shape
result = np.zeros((_m, _n))

for i in range(_m):
    for j in range(_n):
        prod = testr[i][j] * testphi[0][j]
        l_frac = prod / _r_phi_sum[i][0]
        # result[i][j] = frac
        numer = np.square(testn[i][0])
        denom = np.square(testsigma[i][0])
        r_frac = numer / denom
        frac_prod = l_frac * r_frac
        result[i][j] = frac_prod

result

In [ ]:
weight_looped(testr, testn, testphi, testsigma)

In [ ]:
# phi0_counts = np.ones((1, R.shape[1]))

# mono_sigma = NDHistogram(np.sqrt(monoN.counts), monoN.midpoints)
# _R, _monoN, _mono_sigma = cut_low_l(R, monoN, L_cut=0.05, sigma=mono_sigma)

# _r = _R.counts.copy()
# _monon = _monoN.counts.copy()
# _monosigma = _mono_sigma.counts.copy()

# _SC = stopping_criteria_looped(_r, _monon, phi0_counts, _monosigma)
# print(_SC)
# _W = weight_looped(_r, _monon, phi0_counts, _monosigma)
# _phi = next_phi_looped(_r, _monon, phi0_counts, _monosigma)
# print(_phi)
# _SC = stopping_criteria_looped(_r, _monon, _phi, _monosigma)
# print(_SC)

In [ ]:
# sc_tol = 0.01
# dd_tol = 0.01
# max_iters = 500

# phi0_counts = np.ones((1, R.shape[1]))
# _phi = phi0_counts.copy()
# mono_sigma = NDHistogram(np.sqrt(monoN.counts), monoN.midpoints)
# _R, _monoN, _mono_sigma = cut_low_l(R, monoN, L_cut=0.05, sigma=mono_sigma)

# _r = _R.counts.copy()
# _monon = _monoN.counts.copy()
# _monosigma = _mono_sigma.counts.copy()

# iters = 0
# chis = []
# phis = []
# weights = []
# errors = []
# iter_text_len = len(str(max_iters))

# chi_n = stopping_criteria_looped(_r, _monon, _phi, _monosigma)
# chi_last = chi_n
# delta_chi_last = 1
# delta_delta = 1

# while delta_delta > dd_tol:
#     _w = weight_looped(_r, _monon, _phi, _monosigma)
#     _phi = next_phi_looped(_r, _monon, _phi, _monosigma)
#     chi_n = stopping_criteria_looped(_r, _monon, _phi, _monosigma)

#     delta_chi = chi_n - chi_last
#     delta_delta = abs(delta_chi - delta_chi_last)
#     chi_last = chi_n
#     delta_chi_last = delta_chi

#     chis.append(chi_n)
#     phis.append(_phi)
#     weights.append(_w)
#     errors.append(delta_delta)

#     if iters % 10 == 0:
#         print(
#             f"Iter. {iters: {iter_text_len}d}: chi = {chi_n:.3g}, rel_rate = {delta_delta: .3g}"
#         )
#     iters += 1
#     if iters >= max_iters:
#         break

In [ ]:
brute_mono_phi, mono_unfolding_info = unfold_spectrum_looped(R, monoN, L_cut=0.05)
brute_dd_phi, dd_unfolding_info = unfold_spectrum_looped(R, ddN, L_cut=0.05)

In [ ]:
len(mono_unfolding_info["phis"])

In [ ]:
brute_mono_phi_flat = brute_mono_phi.reshape(-1)
brute_dd_phi_flat = brute_dd_phi.reshape(-1)
brute_phi_mids = R.midpoints[1]

In [ ]:
figsize=(9,6)
dd_scaling = 1
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.plot(brute_phi_mids, brute_mono_phi_flat, marker="o", markersize=3, label="2.45 MeV monoenergetic simulation")
ax.plot(brute_phi_mids, brute_dd_phi_flat * dd_scaling, marker="o", markersize=3, label="DD fusion simulation")
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("E (MeV)", fontsize=fontsize)
ax.set_ylabel("Normalized counts", fontsize=fontsize)
ax.set_title("Using Python loop")
# ax.set_yscale("log")
ax.tick_params(labelsize=fontsize)
plt.show()